In [1]:
# === Cell 1: imports & seed ===
!pip -q install pydicom > /dev/null
!pip install opencv-python

import os, glob, math, random, time, shutil, gc
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms

def set_seed(seed=3407):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(3407)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)


device = cuda


In [2]:
# === Cell 2: 讀 train/test + 修 ID_str + 建 DICOM index ===
import os
from pathlib import Path
import glob
import csv
import numpy as np
import pandas as pd

# Kaggle dataset root
DATA_ROOT = Path("/kaggle/input/hw3-data")
DICOM_DIR = DATA_ROOT / "DICOM"

print("[Path] DATA_ROOT =", DATA_ROOT)
print("[Path] DICOM_DIR =", DICOM_DIR)

def read_csv_manual(path):
    rows = []
    with open(path, "r", newline="") as f:
        reader = csv.reader(f)
        for r in reader:
            rows.append(r)
    header = rows[0]
    data   = rows[1:]
    arr = np.array(data, dtype=object)
    df = pd.DataFrame(arr, columns=header)
    return df

# ---------- 讀 train / test ----------
train_df = read_csv_manual(DATA_ROOT / "train.csv")
test_df  = read_csv_manual(DATA_ROOT / "test.csv")

# ---------- 統一欄位名稱 ----------
if "Label" not in train_df.columns and "Disease" in train_df.columns:
    train_df = train_df.rename(columns={"Disease": "Label"})

if "ID" not in train_df.columns:
    for c in ["Id", "id"]:
        if c in train_df.columns:
            train_df = train_df.rename(columns={c: "ID"})

if "ID" not in test_df.columns:
    for c in ["Id", "id"]:
        if c in test_df.columns:
            test_df = test_df.rename(columns={c: "ID"})

# ---------- 修正 ID，去掉 'tensor(12345)' 之類，再轉成 int ----------
train_df["ID"] = (
    train_df["ID"]
    .astype(str)
    .str.replace(r"tensor\((\d+)\)", r"\1", regex=True)
    .astype(int)
)
test_df["ID"] = (
    test_df["ID"]
    .astype(str)
    .str.replace(r"tensor\((\d+)\)", r"\1", regex=True)
    .astype(int)
)

# Label 轉 int
train_df["Label"] = train_df["Label"].astype(int)

# ⭐ 把 ID 變成 7 位字串，和 DICOM 資料夾名稱一樣
train_df["ID_str"] = train_df["ID"].astype(str).str.zfill(7)
test_df["ID_str"]  = test_df["ID"].astype(str).str.zfill(7)

# ---------- 基本資訊 ----------
print("train_df shape:", train_df.shape)
print("test_df  shape:", test_df.shape)
print("train columns:", train_df.columns.tolist())
print("test  columns:", test_df.columns.tolist())

# ---------- 建立 DICOM index (T1 / T2 路徑字典) ----------
dicom_dirs = sorted([p for p in DICOM_DIR.iterdir() if p.is_dir()])
print("\n實際 DICOM 子資料夾數量:", len(dicom_dirs))
print("實際 DICOM 子資料夾名 (前 10 個):", [p.name for p in dicom_dirs[:10]])
print("train/test 的 ID_str (前 10 個):", train_df["ID_str"].tolist()[:10])

DICOM_T1_PATHS = {}
DICOM_T2_PATHS = {}

for d in dicom_dirs:
    sid = d.name   # 例如 '0416567'
    t1_files = sorted(glob.glob(str(d / "T1" / "*.dcm")))
    t2_files = sorted(glob.glob(str(d / "T2" / "*.dcm")))
    if t1_files:
        DICOM_T1_PATHS[sid] = t1_files
    if t2_files:
        DICOM_T2_PATHS[sid] = t2_files

# ---------- 檢查 train / test 是否都能在 DICOM 裡找到 ----------
train_ids_str = train_df["ID_str"].tolist()
test_ids_str  = test_df["ID_str"].tolist()
dicom_ids = set(DICOM_T1_PATHS.keys()) | set(DICOM_T2_PATHS.keys())

train_inter = set(train_ids_str) & dicom_ids
test_inter  = set(test_ids_str)  & dicom_ids

print("\n[Index] train 和 DICOM 的交集數量:", len(train_inter))
print("[Index] test  和 DICOM 的交集數量:", len(test_inter))

missing_train = [i for i in train_ids_str if i not in dicom_ids][:10]
missing_test  = [i for i in test_ids_str  if i not in dicom_ids][:10]
print("train 有哪些 ID_str 沒出現在 DICOM (前幾個):", missing_train)
print("test  有哪些 ID_str 沒出現在 DICOM (前幾個):", missing_test)

# 之後 Dataset / get_subject_volumes 會用到：
# DICOM_T1_PATHS, DICOM_T2_PATHS, train_df, test_df


[Path] DATA_ROOT = /kaggle/input/hw3-data
[Path] DICOM_DIR = /kaggle/input/hw3-data/DICOM
train_df shape: (80, 3)
test_df  shape: (40, 5)
train columns: ['ID', 'Label', 'ID_str']
test  columns: ['ID', 'Disease 0', 'Disease 1 ', 'Disease', 'ID_str']

實際 DICOM 子資料夾數量: 120
實際 DICOM 子資料夾名 (前 10 個): ['0002194', '0002618', '0003158', '0004194', '0004725', '0008098', '0011338', '0026714', '0035791', '0036635']
train/test 的 ID_str (前 10 個): ['0545876', '0517509', '0004194', '0008098', '0230450', '0193329', '0889904', '1126761', '0310883', '0347447']

[Index] train 和 DICOM 的交集數量: 80
[Index] test  和 DICOM 的交集數量: 40
train 有哪些 ID_str 沒出現在 DICOM (前幾個): []
test  有哪些 ID_str 沒出現在 DICOM (前幾個): []


In [4]:
# === Cell 3: DICOM 讀取 + get_subject_volumes（用 ID_str） ===
import torch
import torch.nn.functional as F
import cv2

# 影像尺寸設定（可以照你原本的改）
TARGET_DEPTH = 32
IMG_SIZE_2D  = 224
IMG_SIZE_3D  = 64

def _to_number(x):
    if isinstance(x, (list, tuple)) and len(x) > 0:
        try:
            return float(x[0])
        except Exception:
            return None
    try:
        return float(x)
    except Exception:
        return None

def load_dicom_series(series_dir: Path):
    files = sorted([str(p) for p in series_dir.glob("*.dcm")])
    if not files:
        return None
    metas = []
    for fp in files:
        try:
            dcm = pydicom.dcmread(fp, stop_before_pixels=False, force=True)
            ins = getattr(dcm, "InstanceNumber", None)
            metas.append((ins if ins is not None else 0, fp, dcm))
        except Exception:
            continue
    if not metas:
        return None
    metas.sort(key=lambda t: (t[0], t[1]))
    
    slices = []
    for _, _, dcm in metas:
        try:
            arr = dcm.pixel_array.astype(np.float32)
        except Exception:
            continue
        slope = float(getattr(dcm, "RescaleSlope", 1.0))
        inter = float(getattr(dcm, "RescaleIntercept", 0.0))
        arr = arr * slope + inter
        
        wc = _to_number(getattr(dcm, "WindowCenter", None))
        ww = _to_number(getattr(dcm, "WindowWidth", None))
        if wc is None or ww is None or ww <= 0:
            lo, hi = np.percentile(arr, [0.5, 99.5])
        else:
            lo, hi = wc - ww/2.0, wc + ww/2.0
        arr = np.clip(arr, lo, hi)
        arr = (arr - lo) / (hi - lo) if hi > lo else np.zeros_like(arr, np.float32)
        slices.append(arr.astype(np.float32))
    if not slices:
        return None
    return np.stack(slices, axis=0)   # (D, H, W)

def resize_depth_trilinear(vol: np.ndarray, target_depth: int):
    t = torch.from_numpy(vol)[None, None]   # (1,1,D,H,W)
    t = F.interpolate(t, size=(target_depth, vol.shape[1], vol.shape[2]),
                      mode="trilinear", align_corners=False)
    return t[0,0].numpy()

def resize_hw_slice(img: np.ndarray, size: int):
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)

def representative_slice(vol: np.ndarray, mode: str = "middle"):
    D = vol.shape[0]
    if mode == "varmax":
        return vol.reshape(D, -1).var(axis=1).argmax()
    return vol[D // 2]

def get_subject_volumes(id_str: str):
    """
    這裡 **只用 ID_str** 當 key：
    - DICOM_INDEX[id_str] = (t1_dir, t2_dir)
    """
    if id_str not in DICOM_INDEX:
        return None, None
    t1_dir, t2_dir = DICOM_INDEX[id_str]
    
    def _load(dir_path):
        if dir_path is None or (not dir_path.exists()):
            return None
        vol = load_dicom_series(dir_path)
        if vol is None:
            return None
        return resize_depth_trilinear(vol, TARGET_DEPTH)
    
    vol_t1 = _load(t1_dir)
    vol_t2 = _load(t2_dir)
    return vol_t1, vol_t2

# 簡單檢查一兩筆
sample_tr = train_df.iloc[0]
v1, v2 = get_subject_volumes(sample_tr["ID_str"])
print("\n[Sample check]")
print("Train sample ID_str =", sample_tr["ID_str"],
      "T1:", None if v1 is None else v1.shape,
      "T2:", None if v2 is None else v2.shape)

missing = 0
ok = 0
for _, row in train_df.iterrows():
    v1, v2 = get_subject_volumes(row["ID_str"])
    if (v1 is None) and (v2 is None):
        missing += 1
    else:
        ok += 1
print(f"[Debug] train_df 中，讀不到任何 volume 的 ID 數量 = {missing} / {missing+ok}")



[Sample check]
Train sample ID_str = 0545876 T1: (32, 512, 512) T2: (32, 512, 512)
[Debug] train_df 中，讀不到任何 volume 的 ID 數量 = 0 / 80


In [12]:
# === Cell X: 2D 影像用的 transforms 與 repeat3_channels ===
import torch
import numpy as np
import torchvision.transforms as transforms

def repeat3_channels(x: torch.Tensor) -> torch.Tensor:
    """
    把 1-channel tensor 變成 3-channel，給 EfficientNet / ResNet 用。
    x 通常是 [C,H,W]，ToTensor 之後 C 可能是 1。
    """
    if x.ndim == 2:          # [H,W] -> [1,H,W]
        x = x.unsqueeze(0)
    if x.shape[0] == 1:      # [1,H,W] -> [3,H,W]
        x = x.repeat(3, 1, 1)
    return x

def _train_augs_2d(img_size: int):
    """
    2D 訓練時用的 augmentation
    """
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(repeat3_channels),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(20),
    ])

def _val_augs_2d(img_size: int):
    """
    2D 驗證 / 測試用（不做隨機增強）
    """
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(repeat3_channels),
    ])
# === 3D CNN model: Small3DNet ===
import torch
import torch.nn as nn
import torch.nn.functional as F

class Small3DNet(nn.Module):
    def __init__(self, in_channels=2, num_classes=2):
        super().__init__()
        # [B, C=2, D, H, W]
        self.conv1 = nn.Conv3d(in_channels, 16, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm3d(16)

        self.conv2 = nn.Conv3d(16, 32, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm3d(32)

        self.conv3 = nn.Conv3d(32, 64, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm3d(64)

        self.pool  = nn.MaxPool3d(kernel_size=2, stride=2)

        # Global average pooling 把剩下空間全部壓成 1x1x1
        self.gap   = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.fc    = nn.Linear(64, num_classes)

    def forward(self, x):
        # x: [B, C, D, H, W]
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # -> [B,16,...]
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # -> [B,32,...]
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # -> [B,64,...]

        x = self.gap(x)             # [B,64,1,1,1]
        x = x.view(x.size(0), -1)   # [B,64]
        x = self.fc(x)              # [B, num_classes]
        return x


In [13]:
# === Cell 4: Datasets（全部改成用 ID_str） ===
from torch.utils.data import Dataset

def _get_label_from_row(row):
    if "Label" in row.index:
        return int(row["Label"])
    if "Disease" in row.index:
        return int(row["Disease"])
    return -1

def _train_augs_2d(img_size):
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(repeat3_channels),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(20),
        transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),  # 原本大概 0.85
        transforms.ColorJitter(brightness=0.2, contrast=0.2),     # 新增一點亮度/對比抖動
    ])

def _val_augs_2d(img_size):
    from torchvision import transforms
    def repeat3_channels(t: torch.Tensor) -> torch.Tensor:
        if t.dim() == 2:
            t = t.unsqueeze(0)
        if t.size(0) == 1:
            return t.expand(3, t.size(1), t.size(2))
        return t[:3]
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(repeat3_channels),
        transforms.Resize((img_size, img_size)),
    ])

class SingleSliceDataset(Dataset):
    def __init__(self, df, is_train=True, modality="T1", slice_mode="middle"):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
        self.modality = modality
        self.slice_mode = slice_mode
        self.tf = _train_augs_2d(IMG_SIZE_2D) if is_train else _val_augs_2d(IMG_SIZE_2D)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ID_str = row["ID_str"]
        vol_t1, vol_t2 = get_subject_volumes(ID_str)
        vol = vol_t1 if self.modality == "T1" else vol_t2
        if vol is None:
            vol = vol_t2 if self.modality == "T1" else vol_t1
        if vol is None:
            sl = np.zeros((IMG_SIZE_2D, IMG_SIZE_2D), np.float32)
        else:
            d = vol.shape[0]
            if self.slice_mode == "varmax":
                idx_slice = np.argmax(vol.reshape(d, -1).var(axis=1))
                sl = vol[idx_slice]
            else:
                sl = vol[d // 2]
            sl = resize_hw_slice(sl, IMG_SIZE_2D)
        x = (sl * 255).astype(np.uint8)
        x = self.tf(x)
        y = _get_label_from_row(row)
        return x, y, ID_str

class EarlyFusionDataset(Dataset):
    def __init__(self, df, is_train=True, slice_mode="middle"):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
        self.slice_mode = slice_mode
        from torchvision import transforms
        self.tf_train = transforms.Compose([
            transforms.ToTensor(),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.RandomResizedCrop(IMG_SIZE_2D, scale=(0.85, 1.0)),
        ])
        self.tf_val = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((IMG_SIZE_2D, IMG_SIZE_2D)),
        ])
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ID_str = row["ID_str"]
        vol_t1, vol_t2 = get_subject_volumes(ID_str)
        if vol_t1 is None and vol_t2 is None:
            sl3 = np.zeros((IMG_SIZE_2D, IMG_SIZE_2D, 3), np.uint8)
        else:
            if vol_t1 is None:
                vol_t1 = vol_t2
            if vol_t2 is None:
                vol_t2 = vol_t1
            d1 = vol_t1.shape[0]
            d2 = vol_t2.shape[0]
            if self.slice_mode == "varmax":
                i1 = np.argmax(vol_t1.reshape(d1, -1).var(axis=1))
                i2 = np.argmax(vol_t2.reshape(d2, -1).var(axis=1))
            else:
                i1 = d1 // 2
                i2 = d2 // 2
            sl1 = resize_hw_slice(vol_t1[i1], IMG_SIZE_2D)
            sl2 = resize_hw_slice(vol_t2[i2], IMG_SIZE_2D)
            c3 = np.stack([sl1, np.abs(sl1 - sl2), sl2], axis=-1)
            sl3 = (c3 * 255).astype(np.uint8)
        x = (self.tf_train if self.is_train else self.tf_val)(sl3)
        y = _get_label_from_row(row)
        return x, y, ID_str

class ThreeDVolumeDataset(Dataset):
    def __init__(self, df, is_train=True):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ID_str = row["ID_str"]
        vol_t1, vol_t2 = get_subject_volumes(ID_str)
        D = TARGET_DEPTH
        if vol_t1 is None:
            v1 = np.zeros((D, IMG_SIZE_3D, IMG_SIZE_3D), np.float32)
        else:
            v1 = np.stack([resize_hw_slice(s, IMG_SIZE_3D) for s in vol_t1], axis=0)
        if vol_t2 is None:
            v2 = np.zeros((D, IMG_SIZE_3D, IMG_SIZE_3D), np.float32)
        else:
            v2 = np.stack([resize_hw_slice(s, IMG_SIZE_3D) for s in vol_t2], axis=0)
        x = torch.from_numpy(np.stack([v1, v2], axis=0).astype(np.float32))
        y = _get_label_from_row(row)
        return x, y, ID_str

class LateFusionPairDataset(Dataset):
    def __init__(self, ds1: Dataset, ds2: Dataset):
        assert len(ds1) == len(ds2)
        self.ds1, self.ds2 = ds1, ds2
    def __len__(self):
        return len(self.ds1)
    def __getitem__(self, i):
        x1, y1, id1 = self.ds1[i]
        x2, y2, id2 = self.ds2[i]
        assert id1 == id2
        return (x1, x2), y1, id1


In [14]:
# === Train 4 models with 4-fold CV (EfficientNet-B0 + 3D, with OOF threshold tuning) ===
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.models as tv_models

# -------------------------------------------------
# 0) 基本設定 & 欄位對齊
# -------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

# 保險：Label / ID 欄位名稱統一
if 'Label' not in train_df.columns and 'Disease' in train_df.columns:
    train_df = train_df.rename(columns={'Disease':'Label'})
if 'ID' not in train_df.columns:
    if 'Id' in train_df.columns: train_df = train_df.rename(columns={'Id':'ID'})
    if 'id' in train_df.columns: train_df = train_df.rename(columns={'id':'ID'})
if 'ID' not in test_df.columns:
    if 'Id' in test_df.columns: test_df = test_df.rename(columns={'Id':'ID'})
    if 'id' in test_df.columns: test_df = test_df.rename(columns={'id':'ID'})

train_df['Label'] = train_df['Label'].astype(int)

print("train_df shape:", train_df.shape, "labels mean:", train_df['Label'].mean())
print("test_df  shape:", test_df.shape)

# -------------------------------------------------
# 1) 手動做 4-fold stratified split（不用 sklearn）
# -------------------------------------------------
labels = train_df['Label'].values
idx0 = np.where(labels == 0)[0]
idx1 = np.where(labels == 1)[0]

rng = np.random.default_rng(3407)
rng.shuffle(idx0); rng.shuffle(idx1)

# 各自切成 4 份
folds0 = np.array_split(idx0, 4)
folds1 = np.array_split(idx1, 4)

print("class0 fold lens:", [len(f) for f in folds0])
print("class1 fold lens:", [len(f) for f in folds1])

# -------------------------------------------------
# 2) sampler / class weight
# -------------------------------------------------
def make_sampler(df):
    counts = df['Label'].value_counts().to_dict()
    w = df['Label'].map(lambda y: 1.0 / counts[y]).values
    return WeightedRandomSampler(w, len(w))

def class_weights_from_df(df):
    c = df['Label'].value_counts().to_dict()
    w0 = 1.0 / max(c.get(0,1), 1)
    w1 = 1.0 / max(c.get(1,1), 1)
    s = w0 + w1
    return torch.tensor([w0/s, w1/s], dtype=torch.float32, device=device)

# -------------------------------------------------
# 3) EfficientNet-B0 builder
# -------------------------------------------------
def build_efficientnet_b0(num_classes=2, in_ch=3, pretrained=True):
    """
    建一個 EfficientNet-B0：
      - pretrained=True  用 ImageNet 權重（開網路時 OK）
      - pretrained=False 從頭訓練
    """
    try:
        weights = tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        m = tv_models.efficientnet_b0(weights=weights)
    except Exception as e:
        print("⚠️ 下載預訓練權重失敗，改用隨機初始化:", e)
        m = tv_models.efficientnet_b0(weights=None)

    # 第一層 conv 調成 in_ch
    old_conv = m.features[0][0]
    if old_conv.in_channels != in_ch:
        new_conv = nn.Conv2d(
            in_ch,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=(old_conv.bias is not None),
        )
        m.features[0][0] = new_conv

    # classifier 改成 num_classes
    in_features = m.classifier[1].in_features
    m.classifier[1] = nn.Linear(in_features, num_classes)
    return m.to(device)

# -------------------------------------------------
# 4) 通用訓練 / 預測 function
# -------------------------------------------------
BATCH_2D = 16
BATCH_3D = 4

LR_2D = 1e-4      # 2D 模型學習率
LR_3D = 1e-4      # 3D 模型學習率
EPOCHS_2D = 18    # 每 fold epoch 數（會跑 4 倍）
EPOCHS_3D = 24
NUM_WORKERS = 0   # Kaggle 多工有時候會亂，可改 2 試試

def make_loader(ds, batch_size, sampler=None, shuffle=False):
    return DataLoader(
        ds,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=(sampler is None and shuffle),
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

def train_one_model(model, train_loader, val_loader, epochs, lr, class_weights, tag=""):
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=max(1, epochs // 3), gamma=0.5
    )

    best_state = None
    best_val_loss = float("inf")
    patience = 6
    waited = 0

    for ep in range(1, epochs + 1):
        # -------- train --------
        model.train()
        tr_loss_sum, tr_n, tr_correct = 0.0, 0, 0
        for x, y, _ in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            tr_loss_sum += loss.item() * y.size(0)
            tr_n += y.size(0)
            tr_correct += (logits.argmax(1) == y).sum().item()

        train_loss = tr_loss_sum / max(1, tr_n)
        train_acc = tr_correct / max(1, tr_n)

        # -------- val --------
        model.eval()
        va_loss_sum, va_n, va_correct = 0.0, 0, 0
        with torch.no_grad():
            for x, y, _ in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss = criterion(logits, y)
                va_loss_sum += loss.item() * y.size(0)
                va_n += y.size(0)
                va_correct += (logits.argmax(1) == y).sum().item()

        val_loss = va_loss_sum / max(1, va_n)
        val_acc = va_correct / max(1, va_n)

        prefix = f"[{tag}] " if tag else ""
        print(
            f"{prefix}{ep:02d}/{epochs} "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.3f}  "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}"
        )

        scheduler.step()

        # early stopping（看 val_loss）
        if val_loss < best_val_loss - 1e-3:
            best_val_loss = val_loss
            waited = 0
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience:
                print(f"{prefix}Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    print(f"{tag} best val_loss = {best_val_loss:.6f}")
    return model

@torch.no_grad()
def predict_p1(model, loader, late_pair=None):
    """
    回傳 (ids, p1_list)，用來做 test 上的 ensemble。
    """
    ids, p1_list = [], []
    if late_pair is None:
        model.eval()
        for x, _, ID in loader:
            x = x.to(device)
            logits = model(x)
            p1 = torch.softmax(logits, dim=1)[:, 1]
            ids.extend(list(ID))
            p1_list.extend(p1.cpu().numpy().tolist())
    else:
        m1, m2 = late_pair
        m1.eval(); m2.eval()
        for (x1, x2), _, ID in loader:
            x1, x2 = x1.to(device), x2.to(device)
            logits = 0.5 * (m1(x1) + m2(x2))
            p1 = torch.softmax(logits, dim=1)[:, 1]
            ids.extend(list(ID))
            p1_list.extend(p1.cpu().numpy().tolist())
    return ids, p1_list

@torch.no_grad()
def predict_val_oof(model, loader, late_pair=None):
    """
    給 validation 用：回傳 (ids, labels, p1_list)，之後拿來掃 threshold。
    """
    ids, y_true, p1_list = [], [], []
    if late_pair is None:
        model.eval()
        for x, y, ID in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            p1 = torch.softmax(logits, dim=1)[:, 1]
            ids.extend([int(i) for i in ID])
            y_true.extend(y.cpu().numpy().tolist())
            p1_list.extend(p1.cpu().numpy().tolist())
    else:
        m1, m2 = late_pair
        m1.eval(); m2.eval()
        for (x1, x2), y, ID in loader:
            x1, x2, y = x1.to(device), x2.to(device), y.to(device)
            logits = 0.5 * (m1(x1) + m2(x2))
            p1 = torch.softmax(logits, dim=1)[:, 1]
            ids.extend([int(i) for i in ID])
            y_true.extend(y.cpu().numpy().tolist())
            p1_list.extend(p1.cpu().numpy().tolist())
    return ids, y_true, p1_list

def accumulate_preds(pred_dict, ids, p1_list):
    for i, p in zip(ids, p1_list):
        i_int = int(i)
        if i_int not in pred_dict:
            pred_dict[i_int] = []
        pred_dict[i_int].append(float(p))

def dict_to_csv(pred_dict, out_csv, thr=0.5):
    all_ids = sorted(pred_dict.keys())
    p1_mean = [np.mean(pred_dict[i]) for i in all_ids]
    df = pd.DataFrame({'ID': all_ids, 'p1': p1_mean})
    df['p0'] = 1 - df['p1']
    df['pred'] = (df['p1'] >= thr).astype(int)
    df.to_csv(out_csv, index=False)
    print(f"saved: {out_csv}  (thr={thr:.3f})")

def search_best_thr(oof_df, name, lo=0.3, hi=0.7, steps=41):
    """
    在 OOF validation 上掃 threshold，選 acc 最好的那個。
    """
    best_thr, best_acc = 0.5, 0.0
    for thr in np.linspace(lo, hi, steps):
        pred = (oof_df["p1"] >= thr).astype(int)
        acc = (pred == oof_df["Label"]).mean()
        if acc > best_acc:
            best_acc, best_thr = acc, thr
    print(f"[{name}] best_thr={best_thr:.3f}, OOF_acc={best_acc:.3f}")
    return best_thr

# -------------------------------------------------
# 5) 先把 test 的 Dataset / Loader 建好（fold 之間共用）
# -------------------------------------------------
te_df = test_df.copy()
te_df['Label'] = 0  # dummy

te_single = SingleSliceDataset(te_df, is_train=False,  modality='T1', slice_mode='varmax')
te_loader_single = make_loader(te_single, BATCH_2D, shuffle=False)

te_t1 = SingleSliceDataset(te_df, is_train=False,  modality='T1', slice_mode='varmax')
te_t2 = SingleSliceDataset(te_df, is_train=False,  modality='T2', slice_mode='varmax')
te_loader_late = make_loader(LateFusionPairDataset(te_t1, te_t2), BATCH_2D, shuffle=False)

te_early = EarlyFusionDataset(te_df, is_train=False, slice_mode='varmax')
te_loader_early = make_loader(te_early, BATCH_2D, shuffle=False)

te_3d = ThreeDVolumeDataset(te_df, is_train=False)
te_loader_3d = make_loader(te_3d, BATCH_3D, shuffle=False)

# 用來累積 4 個 fold 的 test p1
single_pred_dict = {}
late_pred_dict   = {}
early_pred_dict  = {}
vol3d_pred_dict  = {}

# 用來存 OOF validation（Label + p1），之後掃 threshold
oof_single_list = []
oof_late_list   = []
oof_early_list  = []
oof_3d_list     = []

# -------------------------------------------------
# 6) 4-fold 訓練 + 對 test 做 ensemble，同時存 OOF val
# -------------------------------------------------
for fold in range(4):
    print("\n" + "="*60)
    print(f"========== Fold {fold+1} / 4 ==========")

    # --- 組這一折的 train / val index ---
    va_idx = np.concatenate([folds0[fold], folds1[fold]])
    tr_idx = np.concatenate(
        [np.concatenate([folds0[i] for i in range(4) if i != fold]),
         np.concatenate([folds1[i] for i in range(4) if i != fold])]
    )
    rng.shuffle(tr_idx); rng.shuffle(va_idx)

    df_tr = train_df.iloc[tr_idx].reset_index(drop=True)
    df_va = train_df.iloc[va_idx].reset_index(drop=True)

    print(f"Fold {fold+1}: Train {len(df_tr)}  Val {len(df_va)}")
    print("  Train label ratio:", df_tr['Label'].mean(),
          " Val label ratio:", df_va['Label'].mean())

    # sampler / class weight 依 fold 重新算
    sampler_tr = make_sampler(df_tr)
    w_cls = class_weights_from_df(df_tr)

    # --- Single slice (T1) ---
    tr_single = SingleSliceDataset(df_tr, is_train=True,  modality='T1', slice_mode='varmax')
    va_single = SingleSliceDataset(df_va, is_train=False, modality='T1', slice_mode='varmax')

    tr_loader_single = make_loader(tr_single, BATCH_2D, sampler=sampler_tr)
    va_loader_single = make_loader(va_single, BATCH_2D, shuffle=False)

    print("\n=== Fold", fold+1, ": Train Single slice (Eff-B0) ===")
    model_single = build_efficientnet_b0(num_classes=2, in_ch=3, pretrained=True)
    model_single = train_one_model(
        model_single, tr_loader_single, va_loader_single,
        epochs=EPOCHS_2D, lr=LR_2D, class_weights=w_cls, tag=f"single-F{fold+1}"
    )
    # test 預測（拿來 ensemble）
    ids, p1 = predict_p1(model_single, te_loader_single)
    accumulate_preds(single_pred_dict, ids, p1)
    # val OOF 預測（拿來掃 threshold）
    ids_v, y_v, p1_v = predict_val_oof(model_single, va_loader_single)
    oof_single_list.append(pd.DataFrame({"ID": ids_v, "Label": y_v, "p1": p1_v}))

    # --- Late fusion (T1+T2) ---
    tr_t1 = SingleSliceDataset(df_tr, is_train=True,  modality='T1', slice_mode='varmax')
    tr_t2 = SingleSliceDataset(df_tr, is_train=True,  modality='T2', slice_mode='varmax')
    va_t1 = SingleSliceDataset(df_va, is_train=False, modality='T1', slice_mode='varmax')
    va_t2 = SingleSliceDataset(df_va, is_train=False, modality='T2', slice_mode='varmax')

    tr_loader_late = make_loader(LateFusionPairDataset(tr_t1, tr_t2), BATCH_2D, sampler=sampler_tr)
    va_loader_late = make_loader(LateFusionPairDataset(va_t1, va_t2), BATCH_2D, shuffle=False)

    print("\n=== Fold", fold+1, ": Train Late fusion (Eff-B0 x2) ===")
    m1 = build_efficientnet_b0(num_classes=2, in_ch=3, pretrained=True)
    m2 = build_efficientnet_b0(num_classes=2, in_ch=3, pretrained=True)
    crit_late = nn.CrossEntropyLoss(weight=w_cls)
    opt1 = torch.optim.AdamW(m1.parameters(), lr=LR_2D, weight_decay=1e-4)
    opt2 = torch.optim.AdamW(m2.parameters(), lr=LR_2D, weight_decay=1e-4)
    sch1 = torch.optim.lr_scheduler.StepLR(opt1, step_size=max(1,EPOCHS_2D//3), gamma=0.5)
    sch2 = torch.optim.lr_scheduler.StepLR(opt2, step_size=max(1,EPOCHS_2D//3), gamma=0.5)

    best_val, best_pair, waited, patience = 1e9, None, 0, 6
    for ep in range(1, EPOCHS_2D+1):
        m1.train(); m2.train()
        tr_loss_sum, tr_n = 0.0, 0
        for (x1,x2), y, _ in tr_loader_late:
            x1,x2,y = x1.to(device), x2.to(device), y.to(device)
            logits1 = m1(x1)
            logits2 = m2(x2)
            logits = 0.5*(logits1 + logits2)
            loss = crit_late(logits, y)
            opt1.zero_grad(); opt2.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(m1.parameters())+list(m2.parameters()), max_norm=5.0)
            opt1.step(); opt2.step()
            tr_loss_sum += loss.item()*y.size(0); tr_n += y.size(0)
        train_loss = tr_loss_sum / max(1,tr_n)

        # val
        m1.eval(); m2.eval()
        v_loss_sum, v_n = 0.0, 0
        with torch.no_grad():
            for (x1,x2), y, _ in va_loader_late:
                x1,x2,y = x1.to(device), x2.to(device), y.to(device)
                logits = 0.5*(m1(x1)+m2(x2))
                loss = crit_late(logits, y)
                v_loss_sum += loss.item()*y.size(0); v_n += y.size(0)
        val_loss = v_loss_sum / max(1,v_n)
        print(f"[late-F{fold+1}] {ep:02d}/{EPOCHS_2D}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")
        sch1.step(); sch2.step()

        if val_loss < best_val - 1e-3:
            best_val = val_loss
            waited = 0
            best_pair = ({k:v.cpu() for k,v in m1.state_dict().items()},
                         {k:v.cpu() for k,v in m2.state_dict().items()})
        else:
            waited += 1
            if waited >= patience:
                print(f"[late-F{fold+1}] Early stopping.")
                break

    if best_pair is not None:
        m1.load_state_dict({k:v.to(device) for k,v in best_pair[0].items()})
        m2.load_state_dict({k:v.to(device) for k,v in best_pair[1].items()})
    print(f"[late-F{fold+1}] best val_loss =", best_val)
    # test ensemble
    ids, p1 = predict_p1(None, te_loader_late, late_pair=(m1,m2))
    accumulate_preds(late_pred_dict, ids, p1)
    # val OOF
    ids_v, y_v, p1_v = predict_val_oof(None, va_loader_late, late_pair=(m1,m2))
    oof_late_list.append(pd.DataFrame({"ID": ids_v, "Label": y_v, "p1": p1_v}))

    # --- Early fusion ---
    tr_early = EarlyFusionDataset(df_tr, is_train=True,  slice_mode='varmax')
    va_early = EarlyFusionDataset(df_va, is_train=False, slice_mode='varmax')

    tr_loader_early = make_loader(tr_early, BATCH_2D, sampler=sampler_tr)
    va_loader_early = make_loader(va_early, BATCH_2D, shuffle=False)

    print("\n=== Fold", fold+1, ": Train Early fusion (Eff-B0) ===")
    model_early = build_efficientnet_b0(num_classes=2, in_ch=3, pretrained=True)
    model_early = train_one_model(
        model_early, tr_loader_early, va_loader_early,
        epochs=EPOCHS_2D, lr=LR_2D, class_weights=w_cls, tag=f"early-F{fold+1}"
    )
    ids, p1 = predict_p1(model_early, te_loader_early)
    accumulate_preds(early_pred_dict, ids, p1)
    ids_v, y_v, p1_v = predict_val_oof(model_early, va_loader_early)
    oof_early_list.append(pd.DataFrame({"ID": ids_v, "Label": y_v, "p1": p1_v}))

    # --- 3D CNN ---
    tr_3d = ThreeDVolumeDataset(df_tr, is_train=True)
    va_3d = ThreeDVolumeDataset(df_va, is_train=False)

    tr_loader_3d = make_loader(tr_3d, BATCH_3D, sampler=sampler_tr)
    va_loader_3d = make_loader(va_3d, BATCH_3D, shuffle=False)

    print("\n=== Fold", fold+1, ": Train 3D CNN ===")
    model_3d = Small3DNet(in_channels=2, num_classes=2).to(device)
    model_3d = train_one_model(
        model_3d, tr_loader_3d, va_loader_3d,
        epochs=EPOCHS_3D, lr=LR_3D, class_weights=w_cls, tag=f"3D-F{fold+1}"
    )
    ids, p1 = predict_p1(model_3d, te_loader_3d)
    accumulate_preds(vol3d_pred_dict, ids, p1)
    ids_v, y_v, p1_v = predict_val_oof(model_3d, va_loader_3d)
    oof_3d_list.append(pd.DataFrame({"ID": ids_v, "Label": y_v, "p1": p1_v}))

# -------------------------------------------------
# 7) OOF validation 合併，掃最佳 threshold
# -------------------------------------------------
oof_single_df = pd.concat(oof_single_list, ignore_index=True)
oof_late_df   = pd.concat(oof_late_list,   ignore_index=True)
oof_early_df  = pd.concat(oof_early_list,  ignore_index=True)
oof_3d_df     = pd.concat(oof_3d_list,     ignore_index=True)

print("\n=== Search best thresholds on OOF validation ===")
thr_single = search_best_thr(oof_single_df, "single")
thr_late   = search_best_thr(oof_late_df,   "late")
thr_early  = search_best_thr(oof_early_df,  "early")
thr_3d     = search_best_thr(oof_3d_df,     "3D")

# -------------------------------------------------
# 8) 4-fold 平均後，輸出最後要交的 single/late/early/3D csv
# -------------------------------------------------
print("\n=== Ensemble 4 folds & save csv (with tuned thresholds) ===")
dict_to_csv(single_pred_dict, "single.csv", thr=thr_single)
dict_to_csv(late_pred_dict,   "late.csv",   thr=thr_late)
dict_to_csv(early_pred_dict,  "early.csv",  thr=thr_early)
dict_to_csv(vol3d_pred_dict,  "3D.csv",     thr=thr_3d)

print("\nDone. Generated (4-fold ensemble): single.csv, late.csv, early.csv, 3D.csv  (ID,p0,p1,pred)")


device = cuda
train_df shape: (80, 3) labels mean: 0.5
test_df  shape: (40, 5)
class0 fold lens: [10, 10, 10, 10]
class1 fold lens: [10, 10, 10, 10]

========== Fold 1 / 4 ==========
Fold 1: Train 60  Val 20
  Train label ratio: 0.5  Val label ratio: 0.5

=== Fold 1 : Train Single slice (Eff-B0) ===
[single-F1] 01/18 train_loss=0.7085 train_acc=0.500  val_loss=0.6689 val_acc=0.700
[single-F1] 02/18 train_loss=0.6909 train_acc=0.533  val_loss=0.6748 val_acc=0.550
[single-F1] 03/18 train_loss=0.6405 train_acc=0.667  val_loss=0.6808 val_acc=0.550
[single-F1] 04/18 train_loss=0.6269 train_acc=0.700  val_loss=0.6933 val_acc=0.500
[single-F1] 05/18 train_loss=0.5963 train_acc=0.733  val_loss=0.7007 val_acc=0.550
[single-F1] 06/18 train_loss=0.5739 train_acc=0.783  val_loss=0.7034 val_acc=0.600
[single-F1] 07/18 train_loss=0.6018 train_acc=0.783  val_loss=0.7059 val_acc=0.650
[single-F1] Early stopping.
single-F1 best val_loss = 0.668887

=== Fold 1 : Train Late fusion (Eff-B0 x2) ===
[late-F

In [15]:
# === Cell 5: Evaluate all four models on validation set (no sklearn/matplotlib) ===
import numpy as np
from torch.utils.data import DataLoader

# ----------- 小工具：confusion matrix + 各種指標自己算 -----------
def binary_confusion_and_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    # confusion matrix: [[TN, FP], [FN, TP]]
    cm = np.zeros((2, 2), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1

    acc = (y_true == y_pred).mean()

    def prf(c):
        tp = np.logical_and(y_pred == c, y_true == c).sum()
        fp = np.logical_and(y_pred == c, y_true != c).sum()
        fn = np.logical_and(y_pred != c, y_true == c).sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        sup  = tp + fn
        return prec, rec, f1, sup

    p0, r0, f10, sup0 = prf(0)
    p1, r1, f11, sup1 = prf(1)
    macro_f1 = (f10 + f11) / 2.0

    # 手動算 AUC
    def binary_auc(y_true, y_score):
        y_true = np.asarray(y_true, dtype=int)
        y_score = np.asarray(y_score, dtype=float)
        n_pos = y_true.sum()
        n_neg = len(y_true) - n_pos
        if n_pos == 0 or n_neg == 0:
            return float("nan")

        order = np.argsort(-y_score)
        y_true_sorted = y_true[order]
        y_score_sorted = y_score[order]

        tp = fp = 0
        tpr = [0.0]
        fpr = [0.0]
        prev_score = None
        for s, y in zip(y_score_sorted, y_true_sorted):
            if prev_score is not None and s != prev_score:
                tpr.append(tp / n_pos)
                fpr.append(fp / n_neg)
            if y == 1:
                tp += 1
            else:
                fp += 1
            prev_score = s
        tpr.append(tp / n_pos)
        fpr.append(fp / n_neg)
        auc = np.trapz(tpr, fpr)
        return auc

    auc = binary_auc(y_true, y_prob)

    metrics = {
        "accuracy": acc,
        "auc": auc,
        "class_0": {"precision": p0, "recall": r0, "f1": f10, "support": sup0},
        "class_1": {"precision": p1, "recall": r1, "f1": f11, "support": sup1},
        "macro_f1": macro_f1,
    }
    return cm, metrics


def eval_2d_model(model, ds_val, name="model", batch_size=None):
    if batch_size is None:
        batch_size = BATCH_SIZE if 'BATCH_SIZE' in globals() else 32
    loader = DataLoader(ds_val, batch_size=batch_size, shuffle=False, num_workers=0)

    y_true, y_pred, y_prob = [], [], []
    model.eval()
    with torch.no_grad():
        for x, y, _ in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            prob = torch.softmax(logits, dim=1)[:, 1]
            pred = (prob >= 0.5).long()
            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(pred.cpu().numpy().tolist())
            y_prob.extend(prob.cpu().numpy().tolist())

    cm, m = binary_confusion_and_metrics(y_true, y_pred, y_prob)
    print(f"\n=== {name} ===")
    print("Confusion matrix (rows=true, cols=pred):")
    print(cm)
    print(f"Accuracy: {m['accuracy']:.4f}, AUC: {m['auc']:.4f}, Macro F1: {m['macro_f1']:.4f}")
    print("Class 0 -> P={:.3f}, R={:.3f}, F1={:.3f}, support={}".format(
        m['class_0']['precision'], m['class_0']['recall'], m['class_0']['f1'], m['class_0']['support']))
    print("Class 1 -> P={:.3f}, R={:.3f}, F1={:.3f}, support={}".format(
        m['class_1']['precision'], m['class_1']['recall'], m['class_1']['f1'], m['class_1']['support']))
    return cm, m


def eval_late_fusion(m1, m2, ds_val_pair, name="Late fusion", batch_size=None):
    if batch_size is None:
        batch_size = BATCH_SIZE if 'BATCH_SIZE' in globals() else 32
    loader = DataLoader(ds_val_pair, batch_size=batch_size, shuffle=False, num_workers=0)

    y_true, y_pred, y_prob = [], [], []
    m1.eval(); m2.eval()
    with torch.no_grad():
        for (x1, x2), y, _ in loader:
            x1, x2, y = x1.to(device), x2.to(device), y.to(device)
            logits = 0.5 * (m1(x1) + m2(x2))
            prob = torch.softmax(logits, dim=1)[:, 1]
            pred = (prob >= 0.5).long()
            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(pred.cpu().numpy().tolist())
            y_prob.extend(prob.cpu().numpy().tolist())

    cm, m = binary_confusion_and_metrics(y_true, y_pred, y_prob)
    print(f"\n=== {name} ===")
    print("Confusion matrix (rows=true, cols=pred):")
    print(cm)
    print(f"Accuracy: {m['accuracy']:.4f}, AUC: {m['auc']:.4f}, Macro F1: {m['macro_f1']:.4f}")
    print("Class 0 -> P={:.3f}, R={:.3f}, F1={:.3f}, support={}".format(
        m['class_0']['precision'], m['class_0']['recall'], m['class_0']['f1'], m['class_0']['support']))
    print("Class 1 -> P={:.3f}, R={:.3f}, F1={:.3f}, support={}".format(
        m['class_1']['precision'], m['class_1']['recall'], m['class_1']['f1'], m['class_1']['support']))
    return cm, m


# ----------- 在這裡重新建立 validation datasets，不吃舊變數名 -----------

# 1) Single-slice validation (T1)
val_single = SingleSliceDataset(df_va, is_train=False, modality='T1', slice_mode='varmax')

# 2) Late fusion validation pair (T1 + T2)
val_t1 = SingleSliceDataset(df_va, is_train=False, modality='T1', slice_mode='varmax')
val_t2 = SingleSliceDataset(df_va, is_train=False, modality='T2', slice_mode='varmax')
val_late_pair = LateFusionPairDataset(val_t1, val_t2)

# 3) Early fusion validation
val_ef = EarlyFusionDataset(df_va, is_train=False, slice_mode='varmax')

# 4) 3D validation
val_3d = ThreeDVolumeDataset(df_va, is_train=False)

# ----------- 實際評估四個模型（前一格訓練完的四個模型物件） -----------
cm_single, m_single = eval_2d_model(model_single, val_single, "Single slice")
cm_late,   m_late   = eval_late_fusion(m1, m2, val_late_pair, "Late fusion")
cm_early,  m_early  = eval_2d_model(model_early, val_ef, "Early fusion")
cm_3d,     m_3d     = eval_2d_model(model_3d, val_3d, "3D CNN (3D volume)")


/tmp/ipykernel_144/292410015.py:60: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc = np.trapz(tpr, fpr)



=== Single slice ===
Confusion matrix (rows=true, cols=pred):
[[2 8]
 [2 8]]
Accuracy: 0.5000, AUC: 0.6700, Macro F1: 0.4505
Class 0 -> P=0.500, R=0.200, F1=0.286, support=10
Class 1 -> P=0.500, R=0.800, F1=0.615, support=10

=== Late fusion ===
Confusion matrix (rows=true, cols=pred):
[[4 6]
 [2 8]]
Accuracy: 0.6000, AUC: 0.6400, Macro F1: 0.5833
Class 0 -> P=0.667, R=0.400, F1=0.500, support=10
Class 1 -> P=0.571, R=0.800, F1=0.667, support=10

=== Early fusion ===
Confusion matrix (rows=true, cols=pred):
[[4 6]
 [3 7]]
Accuracy: 0.5500, AUC: 0.6600, Macro F1: 0.5396
Class 0 -> P=0.571, R=0.400, F1=0.471, support=10
Class 1 -> P=0.538, R=0.700, F1=0.609, support=10

=== 3D CNN (3D volume) ===
Confusion matrix (rows=true, cols=pred):
[[7 3]
 [3 7]]
Accuracy: 0.7000, AUC: 0.8200, Macro F1: 0.7000
Class 0 -> P=0.700, R=0.700, F1=0.700, support=10
Class 1 -> P=0.700, R=0.700, F1=0.700, support=10


In [16]:
# === Cell 6: Count number of parameters for all models ===
def count_params(model):
    return sum(p.numel() for p in model.parameters())

models = {
    "Single slice":        model_single,
    "Late fusion (T1 part)": m1,
    "Late fusion (T2 part)": m2,
    "Early fusion":        model_early,
    "3D CNN":              model_3d,
}

print("Parameter counts:")
for name, model in models.items():
    n = count_params(model)
    print(f"{name:20s} --> {n:,} parameters")


Parameter counts:
Single slice         --> 4,010,110 parameters
Late fusion (T1 part) --> 4,010,110 parameters
Late fusion (T2 part) --> 4,010,110 parameters
Early fusion         --> 4,010,110 parameters
3D CNN               --> 70,450 parameters


In [17]:
# === Cell 7: Generate E3 & Kaggle submission files ===
import pandas as pd
import zipfile
import re

BASE_FILES = ['single.csv', 'late.csv', 'early.csv', '3D.csv']

E3_files = []
KAGGLE_files = []

for f in BASE_FILES:
    df = pd.read_csv(f)

    # 把 'tensor(5008735)' 這種字樣轉成純數字（保險用）
    df['ID'] = df['ID'].astype(str).apply(
        lambda x: re.sub(r'tensor\((\d+)\)', r'\1', x)
    )
    df['ID'] = df['ID'].astype(int)

    # 如果是舊格式只有 'prob'，幫你補出 p0 / p1 / pred
    if 'prob' in df.columns and 'p1' not in df.columns:
        df['p1'] = df['prob']
        df['p0'] = 1 - df['p1']
        df['pred'] = (df['p1'] >= 0.5).astype(int)

    # 確保必備欄位存在
    for col in ['p0', 'p1', 'pred']:
        if col not in df.columns:
            raise ValueError(f"{f} 缺少欄位 {col}，請確認前面 predict 的程式有產生 p0/p1/pred")

    # --- E3 版：ID, p0, p1, pred ---
    e3_df = df[['ID', 'p0', 'p1', 'pred']].copy()
    e3_out = f.replace('.csv', '_E3.csv')
    e3_df.to_csv(e3_out, index=False)
    E3_files.append(e3_out)

    # --- Kaggle 版：ID, pred ---
    kaggle_df = df[['ID', 'pred']].copy()
    kaggle_out = f.replace('.csv', '_KAGGLE.csv')
    kaggle_df.to_csv(kaggle_out, index=False)
    KAGGLE_files.append(kaggle_out)

    print(f"saved: {e3_out} & {kaggle_out}")

# 打包 zip
with zipfile.ZipFile('E3_submission.zip', 'w') as z:
    for f in E3_files:
        z.write(f)

with zipfile.ZipFile('KAGGLE_submission.zip', 'w') as z:
    for f in KAGGLE_files:
        z.write(f)

print("\nDone! Generated:")
print(" - E3_submission.zip (含四個 *_E3.csv)")
print(" - KAGGLE_submission.zip (含四個 *_KAGGLE.csv)")


saved: single_E3.csv & single_KAGGLE.csv
saved: late_E3.csv & late_KAGGLE.csv
saved: early_E3.csv & early_KAGGLE.csv
saved: 3D_E3.csv & 3D_KAGGLE.csv

Done! Generated:
 - E3_submission.zip (含四個 *_E3.csv)
 - KAGGLE_submission.zip (含四個 *_KAGGLE.csv)
